## Учёт заявок в IT-поддержке”

В компании есть небольшая IT-поддержка. Сотрудники отправляют заявки: `не работает принтер`, `забыл пароль`, `нет интернета`. Нужно сделать консольную программу, которая помогает хранить и обрабатывать эти заявки.

### Что должна уметь программа

Меню:
1. Добавить заявку
2. Показать все заявки
3. Изменить статус заявки
4. Показать открытые заявки
5. Найти заявку по имени сотрудника
0. Выход

### Данные одной заявки
```txt
Номер заявки
Имя сотрудника
Описание проблемы
Статус
```

**Статусы можно сделать такими:**
`новая`, `в работе`, `закрыта`


### Пример заявки
```txt
№1
Сотрудник: Иван
Проблема: Не работает принтер
Статус: новая
```

### Обязательные условия
1. Номер заявки создаётся автоматически.
2. При добавлении заявки статус всегда `новая`.
3. Можно изменить статус только на:
    - `новая`
    - `в работе`
    - `закрыта`
4. При поиске по имени программа должна показывать все заявки этого сотрудника.
5. Открытые заявки — это заявки со статусом `новая` или `в работе`.
6. Решение должно быть реализовано в объектно-ориентированном стиле с соблюдением принципов `SOLID`:
   - отдельный класс для заявки;
   - отдельный класс или сервис для управления заявками;
   - отдельный класс для работы с консольным меню;
   - классы не должны брать на себя лишние обязанности;
   - зависимости между частями программы должны быть минимальными и понятными.

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class Ticket:
    number: int
    employee_name: str
    description: str
    status: str

    def __str__(self) -> str:
        return f"№{self.number}\nСотрудник: {self.employee_name}\nПроблема: {self.description}\nСтатус: {self.status}"
    
class TicketService:
    def __init__(self):
        self._tickets: List[Ticket] = []
        self._next_number = 1

    def add_ticket(self, employee_name: str, description: str) -> Ticket:
        ticket = Ticket(
            number=self._next_number,
            employee_name=employee_name,
            description=description,
            status="новая"
        )
        self._tickets.append(ticket)
        self._next_number += 1
        return ticket
    
    def get_all_tickets(self) -> List[Ticket]:
        return self._tickets
    
    def find_by_employee(self, employee_name: str) -> List[Ticket]:
        return [
            ticket for ticket in self._tickets
            if employee_name.lower() in ticket.employee_name.lower()
        ]
    
    def get_open_tickets(self) -> List[Ticket]:
        return [
            ticket for ticket in self._tickets
            if ticket.status in ["новая", "в работе"]
        ]
    
    def update_status(self, ticket_number: int, new_status: str) -> bool:
        valid_statuses = ["новая", "в работе", "закрыта"]
        if new_status not in valid_statuses:
            return False
        
        for ticket in self._tickets:
            if ticket.number == ticket_number:
                ticket.status = new_status
                return True
        return False

class ConsoleMenu:
    def __init__(self, ticket_service: TicketService):
        self.ticket_service = ticket_service

    def show_menu(self):
        print("Учет заявок в IT-поддержке")
        print("1. Добавить заявку")
        print("2. Показать все заявки ")
        print("3. Изменить статус заявки")
        print("4. Показать открытые заявки")
        print("5. Найти заявку по имени сотрудника")
        print("6. Выход")

    def run(self):
        while True:
            self.show_menu()
            choice = input("Выберите пункт меню: ").strip()

            if choice == "1":
                self._add_ticket()
            elif choice == "2":
                self._show_all_tickets()
            elif choice == "3":
                self._update_status()
            elif choice == "4":
                self._show_open_tickets()
            elif choice == "5":
                self._find_by_employee()
            elif choice == "6":
                print("Выход из программы")
            else:
                print("Неверный выбор. Попробуйте снова")

    def _add_ticket(self):
        print("Добовление новой заявки")
        employee_name = input("Имя сотрудника: ").strip()
        description = input("Описание проблемы: ").strip()

        if not employee_name or not description:
            print("Ошибка:  имя сотрудника и описание проблемы не могут быть пустыми")
            return
        
        ticket = self.ticket_service.add_ticket(employee_name, description)
        print(f"Заявка №{ticket.number} успешно добавлена")

    def _show_all_tickets(self):
        tickets = self.ticket_service.get_all_tickets()
        if not tickets:
            print("Заявок пока нет")
            return
        
        print("Все заявки")
        for ticket in tickets:
            print(ticket)

    def _update_status(self):
        try:
            ticket_number = int(input("Введите номер заказа: "))
        except ValueError:
            print("Ошибка: номер заявки должен быть числом")
            return
        
        print("Доступные статнусы: новая, в работе, закрыта")
        new_status = input("Новый статус: ").strip()

        if self.ticket_service.update_status(ticket_number, new_status):
            print("Статус заявки изменен")
        else:
            print("Заявка не найдена или указан неверный статус")

    def _show_open_tickets(self):
        open_tickets = self.ticket_service.get_open_tickets()
        if not open_tickets:
            print("Открытых заявок нет")

        print("Открытые заявки")
        for ticket in open_tickets:
            print(ticket)
        return
    
    def _find_by_employee(self):
        employee_name = input("Введете имя сотрудника: ").strip()
        if not employee_name:
            print("Ошибка: имя сотрудника не может быть пустым")
            return
        

        found_tickets = self.ticket_service.find_by_employee(employee_name)
        if not found_tickets:
            print("Заявка от сотрудника не найдена")

        print("Заявки сотрудника")
        for ticket in found_tickets:
            print(ticket)
            
        return
        
    
def main():
    ticket_service = TicketService()
    menu = ConsoleMenu(ticket_service)
    menu.run()

if __name__ == "__main__":
    main()






Учет заявок в IT-поддержке
1. Добавить заявку
2. Показать все заявки 
3. Изменить статус заявки
4. Показать открытые заявки
5. Найти заявку по имени сотрудника
6. Выход


Выберите пункт меню:  1


Добовление новой заявки


Имя сотрудника:  Фея
Описание проблемы:  хочет спать


Заявка №1 успешно добавлена
Учет заявок в IT-поддержке
1. Добавить заявку
2. Показать все заявки 
3. Изменить статус заявки
4. Показать открытые заявки
5. Найти заявку по имени сотрудника
6. Выход


...

Учет заявок в IT-поддержке
1. Добавить заявку
2. Показать все заявки 
3. Изменить статус заявки
4. Показать открытые заявки
5. Найти заявку по имени сотрудника
6. Выход


Выберите пункт меню:  4


Открытые заявки
№1
Сотрудник: Фея
Проблема: хочет спать
Статус: новая
Учет заявок в IT-поддержке
1. Добавить заявку
2. Показать все заявки 
3. Изменить статус заявки
4. Показать открытые заявки
5. Найти заявку по имени сотрудника
6. Выход


Выберите пункт меню:  5
Введете имя сотрудника:  6


AttributeError: 'Ticket' object has no attribute 'emloyee_name'